# 🎯 Ćwiczenie 2: Formaty danych w skali — CSV vs Parquet na danych z hulajnóg 🛴

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 2
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐ (Podstawowy–Średni)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Porównać **CSV** i **Parquet** pod względem rozmiaru na dysku i czasu odczytu
2. Zmniejszyć zużycie pamięci RAM przez dobór **typów danych** (`category`, `int8`)
3. Stosować **column pruning** i różne kodeki **kompresji** (snappy, gzip, zstd)
4. **Partycjonować** dane (układ Hive-style) i czytać tylko potrzebne partycje
5. Przetwarzać plik większy niż pamięć **w kawałkach** (chunks)
6. Zrównoleglić obliczenia na partycjach — wzorzec *map → reduce*
7. Zrozumieć **problem małych plików** (small files problem)

## 📋 Wymagania wstępne
- Podstawy pandas: `read_csv`, `groupby`, `agg` (z Ćwiczenia 1)
- Pojęcia z Wykładu 2: format wierszowy vs kolumnowy, Parquet, partycjonowanie

---
> 💡 **Tip:** Uruchamiaj każdą komórkę przez `Shift+Enter` lub przycisk ▶️.
> Jeśli utkniesz, sprawdź komórki z podpowiedziami (💡 Hint) albo zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — importy i generowanie danych przejazdów hulajnogami

# pyarrow i duckdb są preinstalowane w Google Colab — ta linia to tylko zabezpieczenie
%pip install -q pyarrow duckdb

import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd

KATALOG = Path("dane_lab2")
if KATALOG.exists():
    shutil.rmtree(KATALOG)  # 👈 czysty start przy każdym uruchomieniu
KATALOG.mkdir()

rng = np.random.default_rng(42)  # 👈 Reprodukowalność — wszyscy dostaną te same dane!
N = 500_000

miasta = ["Warszawa", "Kraków", "Wrocław", "Gdańsk", "Poznań", "Łódź", "Lublin", "Katowice"]
wagi_miast = [0.28, 0.18, 0.12, 0.1, 0.1, 0.09, 0.07, 0.06]
operatorzy = ["Bolt", "Lime", "Dott", "Tier"]

czas_min = rng.gamma(shape=2.0, scale=6.0, size=N).round(1) + 1
df = pd.DataFrame({
    "przejazd_id": np.arange(1, N + 1),
    "data": pd.Timestamp("2026-04-01") + pd.to_timedelta(rng.integers(0, 183 * 24 * 60, N), unit="min"),
    "miasto": rng.choice(miasta, size=N, p=wagi_miast),
    "operator": rng.choice(operatorzy, size=N),
    "typ_uzytkownika": rng.choice(["jednorazowy", "abonament"], size=N, p=[0.7, 0.3]),
    "czas_min": czas_min,
    "dystans_km": (czas_min * rng.uniform(0.12, 0.3, N)).round(2),
    "ocena": rng.choice([1, 2, 3, 4, 5], size=N, p=[0.04, 0.06, 0.15, 0.35, 0.4]),
})
df["kwota_pln"] = (3.0 + df["czas_min"] * 0.79).round(2)  # 👈 opłata startowa + stawka za minutę

print(f"✅ Wygenerowano {len(df):,} przejazdów hulajnogami")
df.head()

## 📖 Szybkie przypomnienie — formaty danych

Na Wykładzie 2 poznaliśmy dwa sposoby zapisu tej samej tabeli:

| Format | Układ | Schemat | Kompresja | Do czego? |
|--------|-------|---------|-----------|-----------|
| **CSV** | wierszowy | ❌ (typy trzeba zgadywać) | słaba | wymiana danych |
| **Parquet** | **kolumnowy** | ✅ zapisany w pliku | świetna (per kolumna) | analityka, Data Lake |

Dziś zmierzysz różnicę **na własne oczy** na 500 000 przejazdów z wypożyczalni hulajnóg w 8 polskich miastach. 🛴

## 🎓 Guided Example — CSV vs Parquet: rozmiar i czas odczytu

**Zadanie:** Zapiszmy te same dane jako CSV i jako Parquet, a potem porównajmy rozmiar plików i czas wczytania.

In [ ]:
# @title 🎓 Guided Example — Uruchom i przeanalizuj

def rozmiar_mb(sciezka: Path) -> float:
    """Zwraca rozmiar pliku lub katalogu (sumę plików) w MB."""
    if sciezka.is_dir():
        return sum(p.stat().st_size for p in sciezka.rglob("*") if p.is_file()) / 1e6
    return sciezka.stat().st_size / 1e6


def zmierz_czas(funkcja) -> float:
    """Zwraca czas wykonania funkcji w sekundach."""
    start = time.perf_counter()
    funkcja()
    return time.perf_counter() - start


plik_csv = KATALOG / "hulajnogi.csv"
plik_parquet = KATALOG / "hulajnogi.parquet"

df.to_csv(plik_csv, index=False)
df.to_parquet(plik_parquet)  # 👈 domyślnie: kompresja snappy

rozmiar_csv_mb = rozmiar_mb(plik_csv)
rozmiar_parquet_mb = rozmiar_mb(plik_parquet)
czas_csv = zmierz_czas(lambda: pd.read_csv(plik_csv))
czas_parquet = zmierz_czas(lambda: pd.read_parquet(plik_parquet))

print(f"📄 CSV:     {rozmiar_csv_mb:6.1f} MB, odczyt {czas_csv:.2f} s")
print(f"🏆 Parquet: {rozmiar_parquet_mb:6.1f} MB, odczyt {czas_parquet:.2f} s")
print(f"👉 Parquet jest {rozmiar_csv_mb / rozmiar_parquet_mb:.1f}× mniejszy i {czas_csv / czas_parquet:.1f}× szybszy w odczycie")

# Bonus: CSV "zapomina" typy danych, Parquet je pamięta
print("\nTyp kolumny 'data' po odczycie z CSV:    ", pd.read_csv(plik_csv, nrows=5)["data"].dtype)
print("Typ kolumny 'data' po odczycie z Parquet:", pd.read_parquet(plik_parquet)["data"].dtype)

---

## ✏️ Ćwiczenie 1: Typy danych a pamięć RAM

**Trudność:** ⭐

W kolumnach tekstowych (`miasto`, `operator`, `typ_uzytkownika`) jest tylko kilka unikalnych wartości powtórzonych setki tysięcy razy — idealni kandydaci na typ `category`. Kolumna `ocena` przyjmuje wartości 1–5, więc `int64` (8 bajtów) to marnotrawstwo — wystarczy `int8` (1 bajt).

1. Policz zużycie pamięci `df` w MB → `pamiec_przed` (użyj `memory_usage(deep=True)`)
2. Utwórz kopię `df_opt`, w której 3 kolumny tekstowe mają typ `category`, a `ocena` typ `int8`
3. Policz zużycie pamięci `df_opt` w MB → `pamiec_po`
4. Policz `oszczednosc = 1 - pamiec_po / pamiec_przed`

**Oczekiwany wynik:** oszczędność powyżej 40%.

In [ ]:
# ✏️ Ćwiczenie 1: Typy danych a pamięć RAM

# TODO: policz pamięć df w MB (deep=True!)
pamiec_przed = ...  # Zastąp ... swoim kodem

df_opt = df.copy()
# TODO: zamień typy kolumn miasto, operator, typ_uzytkownika na "category"
# TODO: zamień typ kolumny ocena na "int8"
...

# TODO: policz pamięć df_opt w MB i oszczędność
pamiec_po = ...  # Zastąp ... swoim kodem
oszczednosc = ...  # Zastąp ... swoim kodem

print(f"Przed: {pamiec_przed:.1f} MB, po: {pamiec_po:.1f} MB, oszczędność: {oszczednosc:.0%}")

In [ ]:
# @title 💡 Hint — Ćwiczenie 1 (kliknij, aby rozwinąć)
# df.memory_usage(deep=True) zwraca Series z bajtami per kolumna — zsumuj ją i podziel przez 1e6.
# deep=True jest kluczowe: bez niego pandas nie liczy rzeczywistego rozmiaru napisów!
# Zmiana typu: df_opt["miasto"] = df_opt["miasto"].astype("category")

In [ ]:
# @title ✅ Test — Ćwiczenie 1 (uruchom, żeby sprawdzić odpowiedź)

try:
    oczekiwane_przed = df.memory_usage(deep=True).sum() / 1e6
    assert abs(pamiec_przed - oczekiwane_przed) < 0.5, "❌ pamiec_przed — czy użyłeś deep=True i podzieliłeś przez 1e6?"
    for kolumna in ["miasto", "operator", "typ_uzytkownika"]:
        assert isinstance(df_opt[kolumna].dtype, pd.CategoricalDtype), f"❌ Kolumna {kolumna} powinna mieć typ category"
    assert df_opt["ocena"].dtype == np.int8, "❌ Kolumna ocena powinna mieć typ int8"
    assert not isinstance(df["miasto"].dtype, pd.CategoricalDtype), "❌ Oryginalny df nie powinien się zmienić — pracuj na kopii!"
    assert oszczednosc > 0.4, f"❌ Oszczędność {oszczednosc:.0%} wydaje się za mała"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 2: Column pruning — czytaj tylko to, czego potrzebujesz

**Trudność:** ⭐

Dyrektor operacyjny pyta: *„Ile zarobiliśmy w każdym mieście?”* Do odpowiedzi potrzebujesz tylko **2 z 10** kolumn.

1. Wczytaj z pliku `plik_parquet` **tylko** kolumny `miasto` i `kwota_pln` → `df_kolumny`
2. Policz przychód per miasto (suma `kwota_pln`), posortowany malejąco → `przychod_miasta` (Series z miastem w indeksie)

**Oczekiwany wynik:** `df_kolumny` ma dokładnie 2 kolumny, a na szczycie `przychod_miasta` jest Warszawa.

In [ ]:
# ✏️ Ćwiczenie 2: Column pruning

# TODO: wczytaj tylko kolumny miasto i kwota_pln z pliku Parquet
df_kolumny = ...  # Zastąp ... swoim kodem

# TODO: policz sumę kwota_pln per miasto, posortuj malejąco
przychod_miasta = ...  # Zastąp ... swoim kodem

czas_wszystkie = zmierz_czas(lambda: pd.read_parquet(plik_parquet))
czas_dwie = zmierz_czas(lambda: pd.read_parquet(plik_parquet, columns=["miasto", "kwota_pln"]))
print(f"Wszystkie kolumny: {czas_wszystkie:.3f} s, dwie kolumny: {czas_dwie:.3f} s")
print(przychod_miasta)

In [ ]:
# @title 💡 Hint — Ćwiczenie 2 (kliknij, aby rozwinąć)
# pd.read_parquet(sciezka, columns=["kolumna1", "kolumna2"])
# Przychód: df_kolumny.groupby("miasto")["kwota_pln"].sum().sort_values(ascending=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 2 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert isinstance(df_kolumny, pd.DataFrame), "❌ df_kolumny powinno być DataFrame"
    assert sorted(df_kolumny.columns) == ["kwota_pln", "miasto"], f"❌ Oczekiwano 2 kolumn, masz: {list(df_kolumny.columns)}"
    assert len(df_kolumny) == len(df), "❌ Liczba wierszy się nie zgadza"
    assert isinstance(przychod_miasta, pd.Series), "❌ przychod_miasta powinno być Series"
    assert przychod_miasta.index[0] == "Warszawa", "❌ Na szczycie powinna być Warszawa — czy posortowałeś malejąco?"
    assert abs(przychod_miasta.sum() - df["kwota_pln"].sum()) < 1, "❌ Suma przychodów się nie zgadza"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 3: Kodeki kompresji

**Trudność:** ⭐⭐

Zapisz `df` do Parquet **czterokrotnie**, za każdym razem z innym kodekiem: `"none"` (bez kompresji), `"snappy"`, `"gzip"`, `"zstd"`. Pliki nazwij `KATALOG / f"hulajnogi_{kodek}.parquet"`.

Zbuduj słownik `rozmiary_mb` (kodek → rozmiar pliku w MB, użyj funkcji `rozmiar_mb`) oraz słownik `czasy_zapisu` (kodek → czas zapisu w sekundach, użyj `zmierz_czas`).

**Oczekiwany wynik:** każdy skompresowany plik jest mniejszy niż `"none"`, a nawet nieskompresowany Parquet jest mniejszy niż CSV.

In [ ]:
# ✏️ Ćwiczenie 3: Kodeki kompresji

kodeki = ["none", "snappy", "gzip", "zstd"]
rozmiary_mb: dict[str, float] = {}
czasy_zapisu: dict[str, float] = {}

for kodek in kodeki:
    sciezka = KATALOG / f"hulajnogi_{kodek}.parquet"
    # TODO: zapisz df do sciezka z kompresją kodek, mierząc czas zapisu
    #       (uwaga: dla "none" przekaż compression=None)
    # TODO: uzupełnij rozmiary_mb[kodek] i czasy_zapisu[kodek]
    ...

for kodek in kodeki:
    print(f"{kodek:>7}: {rozmiary_mb[kodek]:6.1f} MB, zapis {czasy_zapisu[kodek]:.2f} s")
print(f"    CSV: {rozmiar_csv_mb:6.1f} MB")

In [ ]:
# @title 💡 Hint — Ćwiczenie 3 (kliknij, aby rozwinąć)
# kompresja = None if kodek == "none" else kodek
# czasy_zapisu[kodek] = zmierz_czas(lambda: df.to_parquet(sciezka, compression=kompresja))
# rozmiary_mb[kodek] = rozmiar_mb(sciezka)

In [ ]:
# @title ✅ Test — Ćwiczenie 3 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert set(rozmiary_mb) == set(kodeki), f"❌ rozmiary_mb powinien mieć klucze {kodeki}"
    assert set(czasy_zapisu) == set(kodeki), f"❌ czasy_zapisu powinien mieć klucze {kodeki}"
    for kodek in kodeki:
        assert (KATALOG / f"hulajnogi_{kodek}.parquet").exists(), f"❌ Brak pliku dla kodeka {kodek}"
    for kodek in ["snappy", "gzip", "zstd"]:
        assert rozmiary_mb[kodek] < rozmiary_mb["none"], f"❌ {kodek} powinien być mniejszy niż brak kompresji"
    assert rozmiary_mb["none"] < rozmiar_csv_mb, "❌ Nawet nieskompresowany Parquet powinien być mniejszy niż CSV"
    najmniejszy = min(rozmiary_mb, key=rozmiary_mb.get)
    print(f"✅ Wszystkie testy przeszły! Najmniejszy plik daje kodek: {najmniejszy} 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

> 🤔 **Pytanie do przemyślenia:** Który kodek wybrałbyś dla danych czytanych **setki razy dziennie** przez dashboardy, a który dla **archiwum**, które czyta się raz w roku? Zapisz odpowiedź poniżej.

*Twoja odpowiedź:* ...

---

## ✏️ Ćwiczenie 4: Partycjonowanie danych (Hive-style)

**Trudność:** ⭐⭐

Analitycy prawie zawsze filtrują przejazdy **po mieście**. Zapiszmy dane tak, żeby każde miasto miało swój katalog:

```
hulajnogi_partycje/
├── miasto=Gdańsk/...
├── miasto=Kraków/...
└── ...
```

1. Zapisz `df` do katalogu `katalog_partycji` z partycjonowaniem po kolumnie `miasto`
2. Policz liczbę podkatalogów partycji → `liczba_partycji`
3. Wczytaj **tylko** partycję Krakowa (użyj parametru `filters`) → `df_krakow`

**Oczekiwany wynik:** 8 partycji, a `df_krakow` zawiera wyłącznie przejazdy z Krakowa.

In [ ]:
# ✏️ Ćwiczenie 4: Partycjonowanie

katalog_partycji = KATALOG / "hulajnogi_partycje"

# TODO: zapisz df do katalog_partycji z partition_cols=["miasto"]
...

# TODO: policz podkatalogi w katalog_partycji
liczba_partycji = ...  # Zastąp ... swoim kodem

# TODO: wczytaj tylko przejazdy z Krakowa używając filters=[("miasto", "==", "Kraków")]
df_krakow = ...  # Zastąp ... swoim kodem

print(f"Partycji: {liczba_partycji}")
print(f"Przejazdów w Krakowie: {len(df_krakow):,}")
print(sorted(p.name for p in katalog_partycji.iterdir()))

In [ ]:
# @title 💡 Hint — Ćwiczenie 4 (kliknij, aby rozwinąć)
# Zapis:   df.to_parquet(katalog_partycji, partition_cols=["miasto"])
# Liczenie: sum(1 for p in katalog_partycji.iterdir() if p.is_dir())
# Odczyt:  pd.read_parquet(katalog_partycji, filters=[("miasto", "==", "Kraków")])
# Ciekawostka: polskie znaki w nazwach katalogów mogą być zakodowane jak w URL (np. Krak%C3%B3w)!

In [ ]:
# @title ✅ Test — Ćwiczenie 4 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert katalog_partycji.exists(), "❌ Katalog partycji nie istnieje — czy zapisałeś dane?"
    assert liczba_partycji == df["miasto"].nunique(), f"❌ Oczekiwano {df['miasto'].nunique()} partycji, masz {liczba_partycji}"
    assert all(p.name.startswith("miasto=") for p in katalog_partycji.iterdir() if p.is_dir()), "❌ Katalogi powinny mieć format miasto=..."
    assert isinstance(df_krakow, pd.DataFrame), "❌ df_krakow powinno być DataFrame"
    assert len(df_krakow) == (df["miasto"] == "Kraków").sum(), "❌ Liczba przejazdów w Krakowie się nie zgadza"
    assert set(df_krakow["miasto"].astype(str)) == {"Kraków"}, "❌ df_krakow zawiera inne miasta — użyj filters"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 5: Plik większy niż RAM? Czytaj w kawałkach!

**Trudność:** ⭐⭐

Udajmy, że `hulajnogi.csv` ma 40 GB i nie zmieści się w pamięci. Przetwórz go w kawałkach po **100 000 wierszy** (`chunksize=100_000`), wczytując tylko kolumny `miasto` i `kwota_pln` (`usecols=`).

Dla każdego kawałka policz częściowe sumy `kwota_pln` per miasto (**map**), a potem dodaj je do słownika `przychod_chunki` (**reduce**). Policz też, ile było kawałków → `liczba_chunkow`.

**Oczekiwany wynik:** 5 kawałków, a sumy zgadzają się z przychodem z Ćwiczenia 2.

In [ ]:
# ✏️ Ćwiczenie 5: Przetwarzanie w kawałkach

przychod_chunki: dict[str, float] = {}
liczba_chunkow = 0

# TODO: iteruj po pd.read_csv(plik_csv, usecols=[...], chunksize=100_000)
#       - zwiększ liczba_chunkow
#       - policz częściowe sumy per miasto (groupby)
#       - dodaj je do przychod_chunki
...

print(f"Liczba kawałków: {liczba_chunkow}")
for miasto, kwota in sorted(przychod_chunki.items(), key=lambda x: -x[1]):
    print(f"{miasto:>10}: {kwota:>14,.2f} PLN")

In [ ]:
# @title 💡 Hint — Ćwiczenie 5 (kliknij, aby rozwinąć)
# for chunk in pd.read_csv(plik_csv, usecols=["miasto", "kwota_pln"], chunksize=100_000):
#     liczba_chunkow += 1
#     czesciowe = chunk.groupby("miasto")["kwota_pln"].sum()
#     for miasto, kwota in czesciowe.items():
#         przychod_chunki[miasto] = przychod_chunki.get(miasto, 0.0) + kwota

In [ ]:
# @title ✅ Test — Ćwiczenie 5 (uruchom, żeby sprawdzić odpowiedź)

try:
    oczekiwane = df.groupby("miasto")["kwota_pln"].sum()
    assert liczba_chunkow == int(np.ceil(len(df) / 100_000)), f"❌ Oczekiwano {int(np.ceil(len(df) / 100_000))} kawałków, masz {liczba_chunkow}"
    assert set(przychod_chunki) == set(oczekiwane.index), "❌ Brakuje niektórych miast w wyniku"
    for miasto, kwota in oczekiwane.items():
        assert abs(przychod_chunki[miasto] - kwota) < 1, f"❌ Zła suma dla miasta {miasto}"
    print("✅ Wszystkie testy przeszły! Właśnie zrobiłeś mini-MapReduce na jednej maszynie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 6: Równoległe przetwarzanie partycji (map → reduce)

**Trudność:** ⭐⭐⭐

Każda partycja z Ćwiczenia 4 to niezależny kawałek danych — można je przetwarzać **równolegle**, jak węzły klastra!

1. Napisz funkcję `przetworz_partycje(katalog: Path) -> tuple[str, float, int]`, która dla katalogu jednej partycji zwraca `(miasto, suma_kwot, liczba_przejazdow)`. Nazwę miasta odczytaj z nazwy katalogu (`miasto=...`) — pamiętaj o `unquote()`, bo polskie znaki mogą być zakodowane jak w URL.
2. Uruchom ją równolegle na wszystkich katalogach partycji przy pomocy `ThreadPoolExecutor(max_workers=4)` → lista `wyniki` (**map**)
3. Zbuduj z wyników DataFrame `podsumowanie` z kolumnami `miasto`, `przychod`, `liczba_przejazdow`, posortowany malejąco po `przychod` (**reduce**)

> 💡 Dlaczego wątki, a nie procesy? `pyarrow` zwalnia **GIL** podczas czytania Parquet, więc wątki działają tu naprawdę równolegle. Dla obliczeń w czystym Pythonie użyj `ProcessPoolExecutor` (jak na wykładzie).

**Oczekiwany wynik:** 8 wierszy, wartości zgodne z `df.groupby("miasto")`.

In [ ]:
# ✏️ Ćwiczenie 6: Równoległe przetwarzanie partycji

def przetworz_partycje(katalog: Path) -> tuple[str, float, int]:
    """MAP: przetwarza jedną partycję i zwraca (miasto, suma_kwot, liczba_przejazdow)."""
    # TODO: odczytaj nazwę miasta z katalog.name (format "miasto=...") i użyj unquote()
    miasto = ...
    # TODO: wczytaj kolumnę kwota_pln z plików Parquet w tym katalogu
    kwoty = ...
    return miasto, float(kwoty.sum()), len(kwoty)


katalogi = sorted(p for p in katalog_partycji.iterdir() if p.is_dir())

# TODO: uruchom przetworz_partycje równolegle na wszystkich katalogach (ThreadPoolExecutor, max_workers=4)
wyniki = ...  # Zastąp ... swoim kodem

# TODO (REDUCE): zbuduj DataFrame podsumowanie, posortowany malejąco po przychod
podsumowanie = ...  # Zastąp ... swoim kodem

podsumowanie

In [ ]:
# @title 💡 Hint — Ćwiczenie 6 (kliknij, aby rozwinąć)
# miasto = unquote(katalog.name.split("=", 1)[1])
# kwoty = pd.read_parquet(katalog, columns=["kwota_pln"])["kwota_pln"]
# with ThreadPoolExecutor(max_workers=4) as pool:
#     wyniki = list(pool.map(przetworz_partycje, katalogi))
# pd.DataFrame(wyniki, columns=["miasto", "przychod", "liczba_przejazdow"]).sort_values("przychod", ascending=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 6 (uruchom, żeby sprawdzić odpowiedź)

try:
    oczekiwane = df.groupby("miasto").agg(przychod=("kwota_pln", "sum"), liczba_przejazdow=("kwota_pln", "size"))
    assert isinstance(wyniki, list) and len(wyniki) == len(katalogi), "❌ wyniki powinny być listą z jednym elementem na partycję"
    assert isinstance(podsumowanie, pd.DataFrame), "❌ podsumowanie powinno być DataFrame"
    assert list(podsumowanie.columns) == ["miasto", "przychod", "liczba_przejazdow"], "❌ Złe nazwy lub kolejność kolumn"
    assert set(podsumowanie["miasto"]) == set(oczekiwane.index), "❌ Nazwy miast się nie zgadzają — czy użyłeś unquote()?"
    assert podsumowanie["przychod"].is_monotonic_decreasing, "❌ Posortuj malejąco po przychod"
    for _, wiersz in podsumowanie.iterrows():
        assert abs(wiersz["przychod"] - oczekiwane.loc[wiersz["miasto"], "przychod"]) < 1, f"❌ Zły przychód dla {wiersz['miasto']}"
        assert wiersz["liczba_przejazdow"] == oczekiwane.loc[wiersz["miasto"], "liczba_przejazdow"], f"❌ Zła liczba przejazdów dla {wiersz['miasto']}"
    print("✅ Wszystkie testy przeszły! Map → Reduce na partycjach — tak właśnie myśli Spark! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 7: Problem małych plików (small files problem)

**Trudność:** ⭐⭐

Te same dane można zapisać jako **1 duży plik** albo jako **setki malutkich** plików. Sprawdźmy, co to oznacza dla wydajności.

1. Weź pierwsze 100 000 wierszy `df` → `probka`
2. Zapisz `probka` jako jeden plik `KATALOG / "jeden_plik.parquet"`
3. Podziel `probka` na **300** kawałków (`probka.iloc[start:stop]`, po ok. 334 wiersze) i zapisz każdy jako osobny plik w katalogu `katalog_male` (`czesc_000.parquet`, `czesc_001.parquet`, ...)
4. Zmierz czas wczytania jednego pliku (`czas_jeden`) i całego katalogu małych plików (`czas_male`)
5. Policz `spowolnienie = czas_male / czas_jeden`

**Oczekiwany wynik:** wczytanie małych plików jest wyraźnie wolniejsze.

In [ ]:
# ✏️ Ćwiczenie 7: Small files problem

katalog_male = KATALOG / "male_pliki"
katalog_male.mkdir(exist_ok=True)

# TODO: przygotuj próbkę i zapisz ją jako jeden plik
probka = ...  # Zastąp ... swoim kodem
...

# TODO: podziel próbkę na 300 kawałków i zapisz każdy jako osobny plik w katalog_male
...

# TODO: zmierz czasy odczytu i policz spowolnienie
czas_jeden = ...  # Zastąp ... swoim kodem
czas_male = ...  # Zastąp ... swoim kodem
spowolnienie = ...  # Zastąp ... swoim kodem

print(f"1 plik: {czas_jeden:.3f} s | {len(list(katalog_male.glob('*.parquet')))} plików: {czas_male:.3f} s")
print(f"👉 Małe pliki są {spowolnienie:.1f}× wolniejsze")

In [ ]:
# @title 💡 Hint — Ćwiczenie 7 (kliknij, aby rozwinąć)
# probka = df.head(100_000)
# wierszy_na_plik = int(np.ceil(len(probka) / 300))
# for i in range(300):
#     kawalek = probka.iloc[i * wierszy_na_plik:(i + 1) * wierszy_na_plik]
#     kawalek.to_parquet(katalog_male / f"czesc_{i:03d}.parquet")
# czas_male = zmierz_czas(lambda: pd.read_parquet(katalog_male))

In [ ]:
# @title ✅ Test — Ćwiczenie 7 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert len(probka) == 100_000, "❌ probka powinna mieć 100 000 wierszy"
    assert (KATALOG / "jeden_plik.parquet").exists(), "❌ Brak pliku jeden_plik.parquet"
    assert len(list(katalog_male.glob("*.parquet"))) == 300, "❌ W katalogu male_pliki powinno być 300 plików"
    assert len(pd.read_parquet(katalog_male)) == 100_000, "❌ Małe pliki razem powinny mieć 100 000 wierszy"
    assert spowolnienie > 1, "❌ Małe pliki powinny wczytywać się wolniej — sprawdź pomiar czasu"
    print(f"✅ Wszystkie testy przeszły! Spowolnienie: {spowolnienie:.1f}× — a to tylko 300 plików na lokalnym dysku! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

> 🤔 **Pytanie do przemyślenia:** W chmurze (GCS, S3) każde otwarcie pliku to osobne zapytanie HTTP z opóźnieniem ~10–50 ms. Jak duże byłoby spowolnienie dla **300 000** małych plików? Dlaczego zalecany rozmiar pliku w Data Lake to 128 MB – 1 GB?

*Twoja odpowiedź:* ...

---

## 🔥 Zadanie bonusowe: SQL wprost na plikach Parquet z DuckDB

**Trudność:** ⭐⭐⭐

**DuckDB** to analityczna baza danych „w procesie” — potrafi wykonać SQL bezpośrednio na plikach Parquet, bez ładowania ich do pandas. Ta sama idea stoi za **Spark SQL** (Wykład 5) i **BigQuery** (Wykład 11).

Napisz zapytanie SQL na **partycjonowanym** katalogu z Ćwiczenia 4, które dla użytkowników z **abonamentem** zwraca per miasto:
- `liczba_przejazdow` — liczba przejazdów
- `srednia_ocena` — średnia ocena (zaokrąglona do 2 miejsc)
- `przychod` — suma `kwota_pln`

Posortuj malejąco po `przychod`. Wynik zapisz jako pandas DataFrame `wynik_duckdb`.

> 💡 Użyj `read_parquet('ścieżka/**/*.parquet', hive_partitioning = true)` — DuckDB odtworzy kolumnę `miasto` z nazw katalogów.

In [ ]:
# 🔥 Bonus: DuckDB

import duckdb

wzorzec = (katalog_partycji / "**" / "*.parquet").as_posix()

# TODO: napisz zapytanie SQL
zapytanie = f"""
    SELECT ...
    FROM read_parquet('{wzorzec}', hive_partitioning = true)
    ...
"""

wynik_duckdb = duckdb.sql(zapytanie).df()
wynik_duckdb

In [ ]:
# @title ✅ Test — Bonus (uruchom, żeby sprawdzić odpowiedź)

try:
    abon = df[df["typ_uzytkownika"] == "abonament"]
    oczekiwane = abon.groupby("miasto")["kwota_pln"].sum().sort_values(ascending=False)
    assert isinstance(wynik_duckdb, pd.DataFrame), "❌ wynik_duckdb powinno być DataFrame"
    for kolumna in ["liczba_przejazdow", "srednia_ocena", "przychod"]:
        assert kolumna in wynik_duckdb.columns, f"❌ Brakuje kolumny {kolumna}"
    assert len(wynik_duckdb) == len(oczekiwane), "❌ Powinien być jeden wiersz na miasto"
    assert wynik_duckdb["liczba_przejazdow"].sum() == len(abon), "❌ Czy przefiltrowałeś tylko abonamenty?"
    assert np.allclose(wynik_duckdb["przychod"].to_numpy(), oczekiwane.to_numpy(), atol=1), "❌ Przychody (lub sortowanie) się nie zgadzają"
    print("✅ Bonus zaliczony! SQL na plikach bez żadnego serwera bazy danych 🦆🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## 🏁 Podsumowanie

Dziś na 500 000 przejazdów hulajnogami przećwiczyłeś:

| Technika | Co daje? | Gdzie spotkasz ją dalej? |
|----------|----------|--------------------------|
| Parquet zamiast CSV | Mniejsze pliki, szybszy odczyt, zachowane typy | Spark, BigQuery, Data Lake |
| Typy danych (`category`, `int8`) | Kilka razy mniej RAM | pandas, Spark (schematy) |
| Column pruning | Czytasz tylko potrzebne kolumny | Spark SQL, BigQuery (płacisz za bajty!) |
| Kompresja | Kompromis rozmiar ↔ szybkość | Konfiguracja Spark, Data Lake |
| Partycjonowanie | Czytasz tylko potrzebne katalogi | `partitionBy` w Spark, tabele BigQuery |
| Chunki + map → reduce | Dane większe niż RAM | **MapReduce (Wykład 3)** |
| Równoległość na partycjach | Wykorzystanie wielu rdzeni | **Spark (Wykład 4)** |

## 🚀 Co dalej?

Na **Wykładzie 3 (20.10.2026)** przechodzimy z jednej maszyny na **klaster**: ekosystem **Hadoop** — HDFS, MapReduce i YARN. Wzorzec *map → reduce*, który dziś napisałeś ręcznie, zobaczysz tam w wersji na tysiące serwerów! 🐘

## 📚 Dodatkowe materiały
- [Apache Parquet — dokumentacja](https://parquet.apache.org/docs/)
- [pandas — Scaling to large datasets](https://pandas.pydata.org/docs/user_guide/scale.html)
- [DuckDB — Reading Parquet files](https://duckdb.org/docs/data/parquet/overview)